In [184]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [185]:
# "SRR11004103","SRR11004104","SRR11004105","SRR11004118","SRR11004119","SRR11004120","SRR11004130","SRR11004131"
attention = False

In [ ]:
sample_name = 'SRR11004131'

Cov_Mod_path = f'./resources/medium_data/{sample_name}_recenter_Cov_Mod.txt'
sequence_df_path = f'./resources/medium_data/{sample_name}_recenter.fa.csv'
# sample_name = os.path.basename(Cov_Mod_path).split('_')[0]

In [ ]:
# Read input files and normalize sample column
Cov_Mod_df = pd.read_csv(Cov_Mod_path,sep = '\t')
sequence_df = pd.read_csv(sequence_df_path, names = ['sample', 'score1', 'score2', 'struc', 'sequence'])
sequence_df['sample'] = Cov_Mod_df['sample'].str.split('>').str[0]

In [ ]:
# Remove rows containing 'N' from sequence_df and sync with Cov_Mod_df
original_row_count = len(sequence_df)
deleted_samples = sequence_df[sequence_df['sequence'].str.contains('N|n')]['sample'].tolist()

sequence_df = sequence_df[~sequence_df['sequence'].str.contains('N|n')]
Cov_Mod_df = Cov_Mod_df[~Cov_Mod_df['sample'].isin(deleted_samples)]

new_row_count = len(sequence_df)
deleted_rows = original_row_count - new_row_count
print(f"Deleted {deleted_rows} rows. Samples: {deleted_samples}")

删除了 0 行，其中 sample 列的样本为：[]


In [ ]:
# Generate label from sample column and save to CSV
Cov_Mod_df['last_part'] = Cov_Mod_df['sample'].str.split('|').str[-1]
Cov_Mod_df['label'] = Cov_Mod_df['last_part'].map({'same': 0, 'diff': 1})
labels_df = Cov_Mod_df[["sample", "label"]]
labels_output_path = f'./resources/NomalSamples/labels/{sample_name}_labels.csv'
labels_df.to_csv(labels_output_path,index = None)

In [ ]:
# Prepare sample_prepared_df with 4-bit one-hot encoded sequence
sample_prepared_df = Cov_Mod_df.iloc[:, 0:-1-1]
base_to_onehot = {
    'A': [1, 0, 0, 0], 'a': [1, 0, 0, 0],
    'T': [0, 1, 0, 0], 't': [0, 1, 0, 0], 
    'C': [0, 0, 1, 0], 'c': [0, 0, 1, 0],
    'G': [0, 0, 0, 1], 'g': [0, 0, 0, 1]
}

def sequence_to_onehot(sequence):
    onehot = []
    for base in sequence:
        onehot.extend(base_to_onehot.get(base, [0, 0, 0, 0]))
    return onehot

sequence_df['onehot_sequence'] = sequence_df['sequence'].apply(sequence_to_onehot)
sample_prepared_df = pd.concat([sample_prepared_df, sequence_df['onehot_sequence']], axis=1)
sample_prepared_df['onehot_sequence'] = sample_prepared_df['onehot_sequence'].apply(lambda x: ''.join(map(str, x)))

In [ ]:
# Add 3-bit one-hot encoded secondary structure
Struc_to_onehot = {
    '.': [1, 0, 0], '(': [0, 1, 0], ')': [0, 0, 1], 
}

def SS_to_onehot(sequence):
    onehot = []
    for base in sequence:
        onehot.extend(Struc_to_onehot.get(base, [0, 0, 0]))
    return onehot

sequence_df['SS_onehot'] = sequence_df['struc'].apply(SS_to_onehot)
sample_prepared_df = pd.concat([sample_prepared_df, sequence_df['SS_onehot']], axis=1)
sample_prepared_df['SS_onehot'] = sample_prepared_df['SS_onehot'].apply(lambda x: ''.join(map(str, x)))

In [ ]:
if (attention == False):
    # Split strings into columns and save
    sample_prepared_df[['seq' + str(i+1) for i in range(160)]] = sample_prepared_df['onehot_sequence'].apply(lambda x: pd.Series(list(x)))
    sample_prepared_df[['struc' + str(i+1) for i in range(120)]] = sample_prepared_df['SS_onehot'].apply(lambda x: pd.Series(list(x)))
    sample_prepared_df = sample_prepared_df.drop(columns=['onehot_sequence', 'SS_onehot'])
    sample_prepared_output_path = f'./resources/NomalSamples/samples/{sample_name}_sample_prepared.csv'
    sample_prepared_df.to_csv(sample_prepared_output_path, sep = ',', index = None)
    
else:
    # Transpose one-hot sequences for attention models
    def reorder_onehot_sequence(sequence):
        chunks = [sequence[i:i+4] for i in range(0, len(sequence), 4)]
        return ''.join([''.join([chunk[j] for chunk in chunks]) for j in range(4)])
    
    def reorder_SS_onehot(sequence):
        chunks = [sequence[i:i+3] for i in range(0, len(sequence), 3)]
        return ''.join([''.join([chunk[j] for chunk in chunks]) for j in range(3)])

    sample_prepared_df['onehot_sequence'] = sample_prepared_df['onehot_sequence'].apply(reorder_onehot_sequence)
    sample_prepared_df['SS_onehot'] = sample_prepared_df['SS_onehot'].apply(reorder_SS_onehot)
    sample_prepared_df[['seq' + str(i+1) for i in range(160)]] = sample_prepared_df['onehot_sequence'].apply(lambda x: pd.Series(list(x)))
    sample_prepared_df[['struc' + str(i+1) for i in range(120)]] = sample_prepared_df['SS_onehot'].apply(lambda x: pd.Series(list(x)))
    sample_prepared_df = sample_prepared_df.drop(columns=['onehot_sequence', 'SS_onehot'])
    sample_prepared_output_path = f'./resources/SelfAttentionSamples/samples/{sample_name}_sample_prepared.csv'
    sample_prepared_df.to_csv(sample_prepared_output_path, sep = ',', index = None)

/tmp/ipykernel_163016/3923070677.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sample_prepared_df[['seq' + str(i+1) for i in range(160)]] = sample_prepared_df['onehot_sequence'].apply(lambda x: pd.Series(list(x)))
/tmp/ipykernel_163016/3923070677.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sample_prepared_df[['seq' + str(i+1) for i in range(160)]] = sample_prepared_df['onehot_sequence'].apply(lambda x: pd.Series(list(x)))
/tmp/ipykernel_163016/3923070677.py:3: PerformanceWarning: DataFrame is highly fragmented.  Th